# MCMC forecast for $\Lambda$CDM with AP (7 PFS redshift bins)

Bayesian posterior sampling via **NUTS** (BlackJAX) for the 1-loop galaxy
power spectrum multipoles $P_\ell(k)$ ($\ell = 0,2,4$) with the
Alcock-Paczyński effect, using PFS survey specifications across all 7
redshift bins.

All sampled parameters are **centered and whitened** using Fisher-matrix
1-$\sigma$ scales so that the sampler sees approximately $\mathcal{N}(0, I)$
geometry.

**Mock data**: fiducial theory vector + shot noise $P_{\rm shot}/\bar{n}$
on the monopole.

| Bin | $z_{\rm mid}$ | $V$ [(Gpc/$h$)$^3$] | $k_{\rm nl}$ [$h$/Mpc] | $\bar{n}$ [($h$/Mpc)$^3$] |
|-----|---------|------|---------|----------|
| 1   | 0.7     | 0.59 | 0.52    | 3.06e-4  |
| 2   | 0.9     | 0.79 | 0.65    | 9.61e-4  |
| 3   | 1.1     | 0.96 | 0.82    | 9.75e-4  |
| 4   | 1.3     | 1.09 | 1.02    | 6.54e-4  |
| 5   | 1.5     | 1.19 | 1.29    | 3.40e-4  |
| 6   | 1.8     | 2.58 | 1.82    | 2.02e-4  |
| 7   | 2.2     | 2.71 | 2.88    | 3.51e-4  |

## Modules

In [ ]:
import os
from functools import partial

import matplotlib
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import matplotlib.pyplot as plt
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "sans-serif",
    "font.sans-serif": "Computer Modern",
    "font.size": 22})

import numpy as np
from tqdm.auto import tqdm

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax.scipy.linalg import block_diag, inv

from ps_1loop_jax import background as bg

from jaxptpolypol.model import CosmoEmulator, PS1LoopModel
from jaxptpolypol.params import (CosmoParams, SurveyParams,
                                  pack_params, unpack_params,
                                  pack_multibin_params, unpack_multibin_params)
from jaxptpolypol.covariance import gaussian_covariance
from jaxptpolypol.inference import (fisher_matrix, marginalize_fisher,
                                     fixed_and_varied_indices)
from jaxptpolypol.plotting import plot_contours, plot_Gaussian
from jaxptpolypol.theory import (make_pk_ell_fn, compute_fiducial_distances,
                                  make_multipole_projector)
from jaxptpolypol.sampler import (make_transform, make_full_params_fn,
                                   make_gaussian_log_prior, make_log_posterior,
                                   run_nuts, samples_to_physical)

## Emulator and model

In [2]:
model_path = '/Users/nguyenmn/cosmopower-jax-for-pfs/cosmology/jense2024/jense_2023_camb_lcdm/networks/jense_2023_camb_lcdm_Pk_lin.npz'

pklin_emulator = CosmoEmulator(probe='custom_log', emulator_path=model_path)
print("Emulator parameters:", pklin_emulator.parameters)

ps1loop_model = PS1LoopModel(do_irres=True)

Tried to load pickle file from pre-trained model, but failed.
This usually means that you have TF>=2.14, or that you are loading a model that was trained on PCA but loaded with the log (or viceversa), or that you are loading a non-standard model from the cosmopower-organization repo.
Falling back to the dictionary, in case this also fails or does not output the right shape make sure you ran the `convert_tf214.py` script, and that a `.npz` file exists among the trained models, and that you ran `pip install .`. Also make sure that you are asking for the right probe between `custom_log` and `custom_pca`.
Emulator parameters: ['ombh2' 'omch2' 'logA' 'ns' 'h' 'z' 'A_b' 'eta_b' 'logT_AGN']


## Fiducial cosmology

In [3]:
cosmo_dict = {
    'ombh2':    0.02242,
    'omch2':    0.11933,
    'logA':     3.047,
    'ns':       0.9665,
    'h':        0.6766,
    'z':        0.7,
    'A_b':      3.13,
    'eta_b':    0.603,
    'logT_AGN': 7.8,
}
cosmo = CosmoParams(cosmo_dict)
print(cosmo)

CosmoParams:
  ombh2: 0.02242
  omch2: 0.11933
  logA: 3.047
  ns: 0.9665
  h: 0.6766
  z: 0.7
  A_b: 3.13
  eta_b: 0.603
  logT_AGN: 7.8


## Survey specifications

In [4]:
# PFS redshift bins
z_bins   = (0.7,  0.9,  1.1,  1.3,  1.5,  1.8,  2.2)
V_bins   = tuple(v * 1000.**3 for v in (0.59, 0.79, 0.96, 1.09, 1.19, 2.58, 2.71))
knl_bins = (0.52, 0.65, 0.82, 1.02, 1.29, 1.82, 2.88)
n_bins   = (3.06e-4, 9.61e-4, 9.75e-4, 6.54e-4, 3.40e-4, 2.02e-4, 3.51e-4)
n_zbins  = len(z_bins)

print(f"Number of redshift bins: {n_zbins}")
for i, (z, V, knl, nd) in enumerate(zip(z_bins, V_bins, knl_bins, n_bins)):
    print(f"  Bin {i+1}: z={z}, V={V/1e9:.2f} (Gpc/h)^3, knl={knl}, ndens={nd:.2e}")

Number of redshift bins: 7
  Bin 1: z=0.7, V=0.59 (Gpc/h)^3, knl=0.52, ndens=3.06e-04
  Bin 2: z=0.9, V=0.79 (Gpc/h)^3, knl=0.65, ndens=9.61e-04
  Bin 3: z=1.1, V=0.96 (Gpc/h)^3, knl=0.82, ndens=9.75e-04
  Bin 4: z=1.3, V=1.09 (Gpc/h)^3, knl=1.02, ndens=6.54e-04
  Bin 5: z=1.5, V=1.19 (Gpc/h)^3, knl=1.29, ndens=3.40e-04
  Bin 6: z=1.8, V=2.58 (Gpc/h)^3, knl=1.82, ndens=2.02e-04
  Bin 7: z=2.2, V=2.71 (Gpc/h)^3, knl=2.88, ndens=3.51e-04


In [5]:
def b1z(z): return 0.9 + 0.4*z
def b2z(z): return -0.704 - 0.208*z + 0.183*z**2 - 0.00771*z**3
def bG2z(z): return -(2./7.)*(b1z(z) - 1.)
def bGamma3z(z): return (23./42.)*(b1z(z) - 1.)

def Dplusz(z):
    return float(bg.growth_factor(
        cosmo_dict['ombh2'], cosmo_dict['omch2'], cosmo_dict['h'], z, mnu=0.06))

def c0z(z): return 25.*Dplusz(z)**2
def c2z(z): return 25.*Dplusz(z)**2
def c4z(z): return Dplusz(z)**2

In [6]:
surveys = []
for i, (z, knl, nd) in enumerate(zip(z_bins, knl_bins, n_bins)):
    survey_dict = {
        'bias': {'b1': b1z(z), 'b2': b2z(z), 'bG2': bG2z(z), 'bGamma3': bGamma3z(z)},
        'ctr':  {'c0': c0z(z), 'c2': c2z(z), 'c4': c4z(z), 'cfog': knl**(-4)},
        'stoch': {'P_shot': 1.0, 'a0': 0., 'a2': 0.},
        'k_nl': knl, 'ndens': nd,
    }
    surveys.append(SurveyParams(survey_dict))

print(f"Survey params per bin: {len(surveys[0].param_keys)}")

Survey params per bin: 13


## Pack parameters

Parameter layout:
```
params = [ cosmo(9) | survey_bin1(13) | ... | survey_bin7(13) ]
```
Total: 9 + 7 $\times$ 13 = 100 parameters.

In [7]:
packed_params = pack_multibin_params(cosmo, surveys)
n_total = packed_params.shape[0]
n_cosmo = sum(cosmo.param_sizes)
n_survey = len(surveys[0].param_keys)
print(f"Total params: {n_total}  (cosmo {n_cosmo} + {n_zbins} x survey {n_survey})")

Total params: 100  (cosmo 9 + 7 x survey 13)


## Performance configuration

Toggle between **laptop mode** (fast compilation, smaller graphs) and
**server mode** (full resolution, production runs).

**Gauss-Legendre projection** (`n_gl`): The multipole integral
$P_\ell(k) = (2\ell+1)\int_0^1 P(k,\mu)\mathcal{L}_\ell(\mu)\,d\mu$
is computed via pretabulated Gauss-Legendre quadrature.  This gives two wins:

1. It replaces Simpson's rule on 256 $\mu$ points with ~16 GL nodes.
2. It computes $P(k,\mu)$ only once per bin and projects all multipoles at once.

**AP background mode** (`BACKGROUND_MODE`): in multi-bin AP runs,
`"tabulated"` computes $\chi(z)$ once per cosmology on a fixed redshift grid and
linearly interpolates $D_A(z)$ at the survey-bin centers.  This removes the
repeated per-bin Simpson integrations from the AP path and is now the default
for MCMC.

In [ ]:
# ── Choose mode ──────────────────────────────────────────────
LAPTOP_MODE = True   # Set to False for full-resolution production runs
BACKGROUND_MODE = "tabulated"
BACKGROUND_NZ = 256
# ─────────────────────────────────────────────────────────────

if LAPTOP_MODE:
    N_K           = 20    # k-grid points (fewer = smaller XLA graph)
    N_GL          = 16    # Gauss-Legendre nodes for mu projection (None = Simpson)
    MU_NUM        = None  # only used if N_GL is None (Simpson fallback)
    NUM_WARMUP    = 100   # Fisher-seeded → less warmup needed
    NUM_SAMPLES   = 500
    NUM_CHAINS    = 2
    SCAN_CHUNK    = 64
    PARALLEL      = True  # vmap chains on single device
else:
    N_K           = 30
    N_GL          = 16    # GL is preferred even in server mode (faster + more accurate)
    MU_NUM        = None
    NUM_WARMUP    = 300
    NUM_SAMPLES   = 1000
    NUM_CHAINS    = 2
    SCAN_CHUNK    = 128
    PARALLEL      = False

quad_desc = f"GL({N_GL})" if N_GL else f"Simpson({MU_NUM})"
print(f"Mode: {'LAPTOP' if LAPTOP_MODE else 'SERVER'}")
print(f"  k points: {N_K},  quadrature: {quad_desc}")
print(f"  AP background: {BACKGROUND_MODE} (nz={BACKGROUND_NZ})")
print(f"  warmup: {NUM_WARMUP},  samples: {NUM_SAMPLES},  chains: {NUM_CHAINS}")
print(f"  scan_chunk: {SCAN_CHUNK},  parallel_chains: {PARALLEL}")

In [ ]:
Hz_fid, DAz_fid = compute_fiducial_distances(cosmo, z_bins)

pk_fn = make_pk_ell_fn(
    ells=(0, 2, 4),
    pklin_emulator=pklin_emulator,
    ps1loop_model=ps1loop_model,
    cosmo_keys=cosmo.param_keys,
    cosmo_sizes=cosmo.param_sizes,
    survey_keys=surveys[0].param_keys,
    ap=True,
    z_bins=z_bins,
    Hz_fid=Hz_fid,
    DAz_fid=DAz_fid,
    n_gl=N_GL,                # Gauss-Legendre projection
    num=MU_NUM or 256,        # fallback if n_gl is None
    background_mode=BACKGROUND_MODE,
    background_nz=BACKGROUND_NZ,
)
jitted_pk = jax.jit(pk_fn)

# Warm up JIT
ells_tuple = (0, 2, 4)
n_ell = len(ells_tuple)
k = jnp.linspace(5e-3, 0.25, N_K)
n_k = len(k)

jitted_pk(packed_params, k=k).block_until_ready()
quad_str = f"GL({N_GL})" if N_GL else f"Simpson({MU_NUM or 256})"
print(
    f"Theory vector length: {n_zbins * n_ell * n_k}  "
    f"({quad_str}, AP={BACKGROUND_MODE})"
)

## Mock data and covariance

Mock data = fiducial theory + shot noise on the monopole.  The shot noise
contribution is $P_{\rm shot} / \bar{n}$ per bin, added only to $P_0(k)$.

In [10]:
# Fiducial theory (signal only)
theory_fid = jitted_pk(packed_params, k=k)
pk_all = theory_fid.reshape(n_zbins, n_ell, n_k)

# Add shot noise: P_shot / ndens on the monopole (ell=0) for each bin
P_shot = 1.0
shot_noise = jnp.zeros_like(pk_all)
for b in range(n_zbins):
    shot_noise = shot_noise.at[b, 0, :].set(P_shot / n_bins[b])

pk_all_with_shot = pk_all + shot_noise
data = pk_all_with_shot.reshape(-1)

print(f"Shot noise P_shot/ndens per bin:")
for b in range(n_zbins):
    print(f"  Bin {b+1} (z={z_bins[b]}): {P_shot / n_bins[b]:.1f} (Mpc/h)^3")

# Block-diagonal Gaussian covariance (uses signal + shot noise in P0)
dk = k[1] - k[0]
cov_blocks = []
for b in range(n_zbins):
    P0, P2, P4 = pk_all_with_shot[b]
    cov_blocks.append(gaussian_covariance(V_bins[b], k, dk, P0, P2, P4))

cov = block_diag(*cov_blocks)
cov_inv = inv(cov)
print(f"\nCovariance shape: {cov.shape}")

Shot noise P_shot/ndens per bin:
  Bin 1 (z=0.7): 3268.0 (Mpc/h)^3
  Bin 2 (z=0.9): 1040.6 (Mpc/h)^3
  Bin 3 (z=1.1): 1025.6 (Mpc/h)^3
  Bin 4 (z=1.3): 1529.1 (Mpc/h)^3
  Bin 5 (z=1.5): 2941.2 (Mpc/h)^3
  Bin 6 (z=1.8): 4950.5 (Mpc/h)^3
  Bin 7 (z=2.2): 2849.0 (Mpc/h)^3

Covariance shape: (420, 420)


## Fisher matrix (for whitening scales)

We compute the Fisher matrix to obtain per-parameter 1-$\sigma$ widths.
These become the *scales* in the whitening transform so that NUTS sees
approximately unit-variance parameters.

In [11]:
# Jacobian and Fisher
jac = jax.jacfwd(jitted_pk, argnums=0)(packed_params, k=k)
F_full = fisher_matrix(cov, jac)

# Varied / fixed indices (same convention as Fisher notebooks)
fixed_cosmo_idx = [5, 6, 7, 8]         # z, A_b, eta_b, logT_AGN
fixed_survey_offsets = [11, 12]         # k_nl, ndens

fixed_idx, varied_idx = fixed_and_varied_indices(
    n_cosmo, n_survey, n_zbins, fixed_cosmo_idx, fixed_survey_offsets,
)
n_varied = len(varied_idx)

# Marginalized Fisher in the varied subspace
F_varied = marginalize_fisher(F_full, varied_idx)
F_varied_inv = inv(F_varied)

# Whitening scales = marginal 1-sigma from Fisher
fisher_sigma = jnp.sqrt(jnp.diag(F_varied_inv))

# Fiducial values of varied params
fid_varied = packed_params[jnp.array(varied_idx)]

print(f"Varied parameters: {n_varied}")
print(f"Fisher sigma range: [{float(fisher_sigma.min()):.4g}, {float(fisher_sigma.max()):.4g}]")

Varied parameters: 82
Fisher sigma range: [0.004112, 367.2]


## MCMC setup: whitening, priors, log-posterior

The whitening transform maps

$$\theta_{\rm varied} \;\longleftrightarrow\; x = \frac{\theta - \theta_{\rm fid}}{\sigma_{\rm Fisher}}$$

so the sampler explores $x \sim \mathcal{N}(0, I)$ (approximately).
Priors are specified in **physical** parameter space and evaluated inside the
log-posterior closure.

In [12]:
# Whitening transform
to_whitened, to_physical = make_transform(center=fid_varied, scale=fisher_sigma)

# Full-param reconstruction (insert varied values into the fixed template)
full_params_fn = make_full_params_fn(packed_params, varied_idx)

# Cosmological parameter names and their indices within the varied vector
cosmo_param_names = (r'$\omega_b$', r'$\omega_c$', r'$\log A$', r'$n_s$', r'$h$')
varied_cosmo_global = [0, 1, 2, 3, 4]  # global indices of varied cosmo params
cosmo_in_varied = [varied_idx.index(i) for i in varied_cosmo_global]

# Gaussian priors: BBN + weak CMB (same as Fisher notebooks)
prior_entries = [
    (cosmo_in_varied[0], 0.02218, 0.00055),   # BBN on ombh2
    (cosmo_in_varied[3], 0.9649,  0.042),      # ns10 on ns
]
log_prior = make_gaussian_log_prior(n_varied, prior_entries)

# Log-posterior in whitened space
theory_fn = partial(jitted_pk, k=k)
log_post = make_log_posterior(
    theory_fn=theory_fn,
    data=data,
    cov_inv=cov_inv,
    log_prior_fn=log_prior,
    to_physical=to_physical,
    full_params_fn=full_params_fn,
)

# Verify it evaluates at the fiducial (whitened = zeros)
x0 = jnp.zeros(n_varied)
lp0 = log_post(x0)
print(f"log-posterior at fiducial: {float(lp0):.6f}")

log-posterior at fiducial: -723.061932


## NUTS sampling

With whitened parameters the default BlackJAX window adaptation starts from
a good region.  The initial position is $x = 0$ (= fiducial).

The notebook now shows separate **warmup** and **sampling** progress bars.
Sampling progress is updated once per scan chunk (controlled by `SCAN_CHUNK`).

**Mass matrix options** (controlled by `adapt_mass_matrix`, `mass_matrix_type`,
and `initial_inverse_mass_matrix`):

| Choice | Meaning |
|---|---|
| `adapt_mass_matrix=True, mass_matrix_type="diagonal"` | Adapt diagonal mass matrix during warmup (default, fast) |
| `adapt_mass_matrix=True, mass_matrix_type="dense"` | Adapt full mass matrix (captures correlations, O(n²)) |
| `adapt_mass_matrix=False, initial_inverse_mass_matrix=...` | Fix mass matrix (only tune step size); useful with Fisher inverse |

**Fisher-seeded mass matrix**: Since parameters are whitened by Fisher $\sigma$'s,
the ideal inverse mass matrix in whitened space is close to the Fisher inverse
rescaled into that basis,
\[
M^{-1}_{m whitened} = S\,F^{-1}\,S, \qquad S = \mathrm{diag}(\sigma_{m Fisher}).
\]
Passing this as the initial (or fixed) mass matrix gives the sampler
a good approximation to the posterior geometry from the start.

In [ ]:
%%time
rng_key = jax.random.key(42)

# Fisher inverse in whitened space: diag(sigma) @ F^{-1} @ diag(sigma)
S = jnp.diag(fisher_sigma)
F_inv_whitened = S @ F_varied_inv @ S

# Diagonal version (for mass_matrix_type="diagonal")
F_inv_whitened_diag = jnp.diag(F_inv_whitened)

warmup_bar = tqdm(total=NUM_CHAINS, desc="Warmup", position=0)
sample_total = NUM_SAMPLES if PARALLEL else NUM_CHAINS * NUM_SAMPLES
sample_bar = tqdm(total=sample_total, desc="Sampling", position=1)

def on_warmup(chain_idx, num_chains):
    warmup_bar.n = chain_idx
    warmup_bar.refresh()
    if chain_idx == num_chains:
        warmup_bar.close()


def on_sample(chain_idx, num_chains, done, total):
    if PARALLEL:
        sample_bar.n = done
    else:
        sample_bar.n = (chain_idx - 1) * total + done
    sample_bar.refresh()
    if (PARALLEL and done == total) or (
        (not PARALLEL) and chain_idx == num_chains and done == total
    ):
        sample_bar.close()

try:
    samples_w, diagnostics = run_nuts(
        rng_key,
        log_post,
        initial_position=x0,
        num_warmup=NUM_WARMUP,
        num_samples=NUM_SAMPLES,
        num_chains=NUM_CHAINS,
        # --- Option 1: Fixed diagonal Fisher mass matrix (fast, no adaptation) ---
        adapt_mass_matrix=False,
        mass_matrix_type="diagonal",
        initial_inverse_mass_matrix=F_inv_whitened_diag,
        # --- Option 2: Fixed dense Fisher mass matrix (captures correlations) ---
        # adapt_mass_matrix=False,
        # mass_matrix_type="dense",
        # initial_inverse_mass_matrix=F_inv_whitened,
        # --- Option 3: Full adaptation from identity (no Fisher seeding) ---
        # adapt_mass_matrix=True,
        # mass_matrix_type="diagonal",
        scan_chunk_size=SCAN_CHUNK,
        parallel_chains=PARALLEL,
        progress_fn=on_warmup,
        sample_progress_fn=on_sample,
    )
finally:
    warmup_bar.close()
    sample_bar.close()

print(f"\nSamples shape: {samples_w.shape}  (chains, samples, params)")

## Diagnostics

In [ ]:
accept = diagnostics["acceptance_rate"]
n_steps = diagnostics["num_integration_steps"]
divergent = diagnostics["is_divergent"]

print("Per-chain diagnostics:")
for c in range(samples_w.shape[0]):
    print(f"  Chain {c+1}:  accept = {float(accept[c].mean()):.3f}"
          f"   steps = {float(n_steps[c].mean()):.1f}"
          f"   divergent = {int(divergent[c].sum())}")

In [ ]:
# Trace plots for cosmological parameters
fig, axes = plt.subplots(len(cosmo_in_varied), 1,
                          figsize=(12, 2.5*len(cosmo_in_varied)),
                          constrained_layout=True)

for i, (ci, name) in enumerate(zip(cosmo_in_varied, cosmo_param_names)):
    ax = axes[i]
    for c in range(samples_w.shape[0]):
        ax.plot(samples_w[c, :, ci], alpha=0.6, lw=0.3, label=f'chain {c+1}')
    ax.set_ylabel(f'{name} (whitened)')
    ax.axhline(0, color='k', ls='--', lw=0.5)
    if i == 0:
        ax.legend(fontsize=8, ncol=2)
axes[-1].set_xlabel('Sample index')
fig.suptitle('Trace plots (whitened space)', fontsize=14)

## Posterior in physical space

In [ ]:
samples_phys = samples_to_physical(samples_w, to_physical)
flat_samples = samples_phys.reshape(-1, n_varied)

# Sample mean and covariance
sample_mean = jnp.mean(flat_samples, axis=0)
sample_cov  = jnp.cov(flat_samples.T)

print("Cosmo posterior summary:")
for name, ci in zip(cosmo_param_names, cosmo_in_varied):
    fid = float(fid_varied[ci])
    mu  = float(sample_mean[ci])
    sig = float(jnp.sqrt(sample_cov[ci, ci]))
    print(f"  {name:15s}  fid = {fid:.5g}   mean = {mu:.5g}   std = {sig:.5g}")

## Corner plot: MCMC vs Fisher

Blue histograms / scatter = MCMC samples.
Red dashed ellipses = Fisher forecast (with same priors).

In [ ]:
from jaxptpolypol.inference import gaussian_prior_fisher, build_prior_sigmas

# Fisher + prior for comparison
prior_sigmas = build_prior_sigmas(
    cosmo_keys=cosmo.param_keys,
    cosmo_sizes=cosmo.param_sizes,
    survey_keys=surveys[0].param_keys,
    n_bins=n_zbins,
    cosmo_priors={'ombh2': 0.00055, 'ns': 0.042},
)
F_prior = gaussian_prior_fisher(n_total, prior_sigmas)
F_with_prior = F_full + F_prior
F_varied_prior = marginalize_fisher(F_with_prior, varied_idx)

In [ ]:
n_plot = len(cosmo_param_names)
ci = np.array(cosmo_in_varied)

plt.rcParams.update({"font.size": 10})
fig = plt.figure(figsize=(n_plot*2.8, n_plot*2.8), constrained_layout=True)

for i in range(n_plot):
    for j in range(n_plot):
        ax = plt.subplot(n_plot, n_plot, i*n_plot + j + 1)
        if j < i:
            # MCMC scatter
            ax.scatter(flat_samples[:, ci[j]], flat_samples[:, ci[i]],
                       s=0.3, alpha=0.15, color='steelblue', rasterized=True)
            # Fisher ellipses (data + prior)
            plot_contours(F_varied_prior, fid_varied, np.array([ci[j], ci[i]]),
                          fill=False, color='crimson', ls='--', lw=1.5,
                          label='Fisher + prior' if (i == 1 and j == 0) else None)
            if i == 1 and j == 0:
                handles, labels = ax.get_legend_handles_labels()
                seen = {}
                for h, l in zip(handles, labels):
                    if l not in seen:
                        seen[l] = h
                ax.legend(seen.values(), seen.keys(), fontsize=7)
        elif j == i:
            # MCMC histogram
            ax.hist(flat_samples[:, ci[i]], bins=40, density=True,
                    color='steelblue', alpha=0.5, edgecolor='none')
            # Fisher Gaussian
            plot_Gaussian(F_varied_prior, fid_varied, ci[i],
                          color='crimson', ls='--', lw=1.5)
        else:
            ax.axis('off')
        if i == n_plot - 1:
            ax.set_xlabel(cosmo_param_names[j])
        if j == 0:
            ax.set_ylabel(cosmo_param_names[i])

fig.suptitle(r'MCMC (blue) vs Fisher (red dashed) — $\Lambda$CDM with AP', fontsize=14)

## $1\sigma$ comparison: MCMC vs Fisher

In [ ]:
sigma_fisher = jnp.sqrt(jnp.diag(inv(F_varied_prior)))

print(f"{'Parameter':>15s}  {'Fiducial':>10s}  {'Fisher':>10s}  {'MCMC':>10s}  {'Ratio':>8s}")
print("-" * 60)
for name, idx in zip(cosmo_param_names, cosmo_in_varied):
    fid  = float(fid_varied[idx])
    sf   = float(sigma_fisher[idx])
    sm   = float(jnp.sqrt(sample_cov[idx, idx]))
    print(f"{name:>15s}  {fid:10.5g}  {sf:10.4g}  {sm:10.4g}  {sm/sf:8.3f}")